# <font color = 'red'> Dependencias

In [66]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio, count_by_category, proportions_by_category, counts_and_proportions_by_category)
from visualization_tools import plot_interactive_chart, plot_categorical_proportions
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> Carga de Datos

In [67]:
col = "Payment_Behaviour"

In [68]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [69]:
df = df.filter(regex = col + '|Credit_Mix|Credit_Score').drop('Binary_Credit_Score', axis = 1)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column                                              Non-Null Count   Dtype 
---  ------                                              --------------   ----- 
 0   Credit_Mix                                          100000 non-null  object
 1   Payment_Behaviour                                   100000 non-null  object
 2   Credit_Score                                        100000 non-null  int64 
 3   Payment_Behaviour_High_spent_Large_value_payments   100000 non-null  bool  
 4   Payment_Behaviour_High_spent_Medium_value_payments  100000 non-null  bool  
 5   Payment_Behaviour_High_spent_Small_value_payments   100000 non-null  bool  
 6   Payment_Behaviour_Low_spent_Large_value_payments    100000 non-null  bool  
 7   Payment_Behaviour_Low_spent_Medium_value_payments   100000 non-null  bool  
 8   Payment_Behaviour_Low_spent_Small_value_payments    100000 non-null  bool  


In [70]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
Duplicate rows found: 99982


In [71]:
# Filtrar solo las filas que están duplicadas
duplicated_rows = df[df.duplicated(keep=False)]

duplicated_rows_sorted = duplicated_rows.sort_values(by=duplicated_rows.columns.tolist())
display(duplicated_rows_sorted.head(5))

,Credit_Mix,Payment_Behaviour,Credit_Score,Payment_Behaviour_High_spent_Large_value_payments,Payment_Behaviour_High_spent_Medium_value_payments,Payment_Behaviour_High_spent_Small_value_payments,Payment_Behaviour_Low_spent_Large_value_payments,Payment_Behaviour_Low_spent_Medium_value_payments,Payment_Behaviour_Low_spent_Small_value_payments
79,Bad,High_spent_Large_value_payments,0,True,False,False,False,False,False
96,Bad,High_spent_Large_value_payments,0,True,False,False,False,False,False
222,Bad,High_spent_Large_value_payments,0,True,False,False,False,False,False
340,Bad,High_spent_Large_value_payments,0,True,False,False,False,False,False
395,Bad,High_spent_Large_value_payments,0,True,False,False,False,False,False


# <font color = 'red'> Análisis

## <font color = 'skyblue'> ANÁLISIS GENERAL

In [48]:
category_names = [i for i in df.drop(col, axis = 1).columns if i.startswith(col)]

results = counts_and_proportions_by_category(df, category_names, "Credit_Mix")
counts_df = results["counts"]
proportions_df = results["proportions"]
summary_df = results["summary"]

counts_df.index = counts_df.index.str.replace(col + '_', '')
proportions_df.index = proportions_df.index.str.replace(col + '_', '')
summary_df.index = summary_df.index.str.replace(col + '_', '')

In [77]:
custom_colors = {
    "prop_Bad": "orangered",
    "prop_Standard": "gold",
    "prop_Good": "skyblue"
}

fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Bad", "prop_Standard", "prop_Good"],  
    stacked=False,                      
    title=f"Proporción de Credit Score por {col}",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1100,
    height=900,
    sort_order='asc',          
    sort_by="prop_Bad"          
)

fig.show()


fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Bad"],  
    stacked=False,                      
    title=f"Proporción de Bad por {col}",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1100,
    height=900,
    sort_order='asc',          
    sort_by="prop_Bad"          
)

fig.show()

fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Good"],  
    stacked=False,                      
    title=f"Proporción Good por {col}",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1100,
    height=900,
    sort_order='asc',          
    sort_by="prop_Good"          
)

fig.show()

fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Standard"],  
    stacked=False,                      
    title=f"Proporción Standard por {col}",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1100,
    height=900,
    sort_order='asc',          
    sort_by="prop_Standard"          
)

fig.show()

# A simple vista sí parece que el comportamiento de pago sí está relacionado con el impago:
#     * La muestra de deudores que no pagan el monto mínimo o son Good (75%) o Standard (25%) y no hay malos deudores.
#     * Por su parte, la muestra de deudores que sí pagan el monto mínimo o son Bad (40%) o Standard (60%) pero no hay malos.
# De modo que no hay deudores buenos que paguen el monto mínimo. Ni malos que no lo paguen. Los Standard, en cambio, están en las dos clases
# pero principalmente en los que pagan el monto mínimo.


<font color = 'skyblue'> Regresión

Los clientes que no pagan solo el monto mínimo tienen un Credit_Score promedio de 1.749, lo que equivale a una calificación cercana a Standard/Good.

Los clientes que sí pagan solo el monto mínimo tienen un Credit_Score promedio de 0.6001, lo que corresponde a una calificación entre Bad/Standard.

In [63]:
y_col = "Credit_Score"

df_ = df[[col]]

model = smf.ols(f"{y_col} ~ " + ' + '.join(df_.columns) + ' -1', data=df).fit() # f"{y_col} ~ " + ' + '.join(df_.columns)

print(model.summary())


                            OLS Regression Results                            
Dep. Variable:           Credit_Score   R-squared:                       0.036
Model:                            OLS   Adj. R-squared:                  0.036
Method:                 Least Squares   F-statistic:                     752.7
Date:                Fri, 11 Apr 2025   Prob (F-statistic):               0.00
Time:                        10:03:28   Log-Likelihood:            -1.0897e+05
No. Observations:              100000   AIC:                         2.180e+05
Df Residuals:                   99994   BIC:                         2.180e+05
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------

In [52]:
y_column = "Credit_Score"
base_category = "Payment_Behaviour_Low_spent_Small_value_payments"

x_columns = [i for i in df.columns if col + '_' in i] # categories

x_columns.remove(base_category)

model = OrderedModel(df[y_column], df[x_columns], distr="logit")

result = model.fit(method='bfgs')

print(result.summary())

Optimization terminated successfully.
         Current function value: 1.042697
         Iterations: 32
         Function evaluations: 34
         Gradient evaluations: 34
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0427e+05
Model:                   OrderedModel   AIC:                         2.086e+05
Method:            Maximum Likelihood   BIC:                         2.086e+05
Date:                Fri, 11 Apr 2025                                         
Time:                        09:59:54                                         
No. Observations:              100000                                         
Df Residuals:                   99993                                         
Df Model:                           5                                         
                                                         coef    std err          z      P>|z|      [0.025      0.975

No pagar el monto mínimo se asocia con mejores categorías de crédito

Puntualmente, pasar de 0 a 1 en Payment_of_Min_Amount (de sí a no) incrementa la probabilidad de que Credit_Score esté en una categoría superior (de Bad a Standard, o de Standard a Good). 

Estar a la derecha de -0.4058 aumenta la probabilidad de pasar de Bad a Standard o Good.

Estar a la derecha de 2.9153 aumenta la probabilidad de pertenecer a la categoría Good.